# Silver - RF-1 Soft-Soil Susceptibility

Clips the bronze-catalogued sources to the pipeline-selected detailed AOI, aligns everything to a **10 m grid in the local UTM zone**, computes RF-1 factor metrics, and scores per-pixel **Susceptibility (S)** and **Consequence (C)** ratings.

| | |
| --- | --- |
| **Reads (bronze)** | Planetary Computer rasters and BC Soil Information Finder Tool survey polygons |
| **Writes (silver)** | `silver_rf1_soil_susceptibility`, one row per 10 m pixel with factor metrics, soil signal, and S/C ratings |
| **Feeds** | `gold_rf1_risk_matrix`, where `risk_score = S * C` |

BC soil survey polygons provide a strong local ground-truth signal where coverage exists. Outside their published coverage, the model continues with available raster signals and records the source gap. Run top-to-bottom with `silver_lakehouse` attached as the default lakehouse.

## Dependencies

Library dependencies are supplied by the **geohazard_env** Fabric Environment attached to this notebook, not by inline `%pip install`. Inline installation is disabled in many tenants and fails with MagicUsageError when a notebook runs from a pipeline. See `fabric/environment/requirements.txt`.

## 2. Area of interest

The pipeline supplies the centre point and detailed analysis radius. This notebook builds a deterministic 10 m grid in the point's local UTM zone so every raster source aligns pixel-for-pixel. The analysis radius is limited to 5 km to keep the demo workload bounded; the wider bronze catalog radius remains independent.

In [ ]:
# Pipeline parameters
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 3.0
PIPELINE_RUN_ID = ""


In [ ]:
# Run identity. Blank values get a unique manual-run identifier, sanitised the same way
# as the gold notebook so silver, gold, and the handoff artefacts share one run_id.
import re
import uuid
from datetime import datetime, timezone

_raw_run_id = str(PIPELINE_RUN_ID or "").strip()
if not _raw_run_id:
    _raw_run_id = f"manual-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"
RUN_ID = re.sub(r"[^A-Za-z0-9._-]+", "-", _raw_run_id)[:128].strip(".-_")
if not RUN_ID:
    raise ValueError("PIPELINE_RUN_ID did not contain any file-system-safe characters.")
print(f"run_id: {RUN_ID}")

In [ ]:
import numpy as np
from datetime import datetime, timezone
from pyproj import Transformer
from affine import Affine
from odc.geo.geobox import GeoBox

LAT = float(LATITUDE)
LON = float(LONGITUDE)
BUFFER_KM = float(RADIUS_KM)

if not -80.0 <= LAT <= 84.0:
    raise ValueError("LATITUDE must be between -80 and 84 degrees for UTM analysis.")
if not -180.0 <= LON <= 180.0:
    raise ValueError("LONGITUDE must be between -180 and 180 degrees.")
if not 0.0 < BUFFER_KM <= 5.0:
    raise ValueError("RADIUS_KM must be greater than 0 and no more than 5 km for the 10 m demo grid.")

AOI_NAME = f"AOI {LAT:.4f}, {LON:.4f}"
RES_M = 10
SEASON = "2024-06-01/2024-09-30"
utm_zone = min(60, max(1, int((LON + 180.0) // 6.0) + 1))
utm_epsg = (32600 if LAT >= 0 else 32700) + utm_zone
UTM_CRS = f"EPSG:{utm_epsg}"

# Project the centre point to its local UTM zone and build a deterministic square geobox.
_to_utm = Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True)
_to_wgs = Transformer.from_crs(UTM_CRS, "EPSG:4326", always_xy=True)
cx, cy = _to_utm.transform(LON, LAT)
half_m = BUFFER_KM * 1000.0

# Snap the origin to the RES_M grid so the box is deterministic.
minx = float(np.floor((cx - half_m) / RES_M) * RES_M)
maxy = float(np.ceil((cy + half_m) / RES_M) * RES_M)
W = int(round((2 * half_m) / RES_M))
H = W
GEOBOX = GeoBox((H, W), Affine(RES_M, 0.0, minx, 0.0, -RES_M, maxy), UTM_CRS)

# WGS84 bbox (minx, miny, maxx, maxy) used for the STAC searches.
_ll = _to_wgs.transform(minx, maxy - H * RES_M)
_ur = _to_wgs.transform(minx + W * RES_M, maxy)
BBOX_WGS = [_ll[0], _ll[1], _ur[0], _ur[1]]

print(f"AOI            : {AOI_NAME}")
print(f"Clip buffer    : {BUFFER_KM} km  ->  {2 * BUFFER_KM} x {2 * BUFFER_KM} km box")
print(f"Analysis grid  : {H} x {W} px @ {RES_M} m  ({UTM_CRS})")
print(f"STAC bbox WGS84: {[round(value, 5) for value in BBOX_WGS]}")
print(f"Season         : {SEASON}")

## 3. Bronze lineage

The bronze tables hold STAC metadata rather than pixels. This step confirms which catalogued scenes intersect the selected detailed analysis extent, then reloads the corresponding COG pixels in the next step.

In [ ]:
from pyspark.sql import functions as F

# Portable OneLake path: resolve immutable workspace and lakehouse IDs at runtime.
WS = notebookutils.runtime.context["currentWorkspaceId"]
BRONZE_LH_NAME = "bronze_lakehouse"
BRONZE_LH_ID = notebookutils.lakehouse.get(BRONZE_LH_NAME, workspaceId=WS).id
ONELAKE_ENDPOINT = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
BRONZE_ABFSS = f"abfss://{WS}@{ONELAKE_ENDPOINT}/{BRONZE_LH_ID}/Tables"

bronze_satellite = {
    "sentinel-2-l2a": "bronze_sentinel_2_l2a",
    "cop-dem-glo-30": "bronze_cop_dem_glo_30",
    "sentinel-1-rtc": "bronze_sentinel_1_rtc",
    "esa-worldcover": "bronze_esa_worldcover",
    "io-lulc-9-class": "bronze_io_lulc_9_class",
    "alos-palsar-mosaic": "bronze_alos_palsar_mosaic",
    "hgb": "bronze_hgb",
}

mnx, mny, mxx, mxy = BBOX_WGS
lineage = []
for collection, table_name in bronze_satellite.items():
    try:
        dataframe = spark.read.format("delta").load(f"{BRONZE_ABFSS}/{table_name}")
        overlap = dataframe.filter(
            (F.col("bbox_minx") <= mxx) & (F.col("bbox_maxx") >= mnx) &
            (F.col("bbox_miny") <= mxy) & (F.col("bbox_maxy") >= mny)
        )
        lineage.append((collection, table_name, dataframe.count(), overlap.count()))
    except Exception as error:
        lineage.append((collection, table_name, -1, -1))
        print(f"  (skip {table_name}: {str(error)[:80]})")

overlap_label = f"overlap_{BUFFER_KM:g}km"
print(f"{'collection':<20}{'bronze_items':>14}{overlap_label:>14}")
for collection, table_name, total, overlap_count in lineage:
    total_text = "n/a" if total < 0 else total
    overlap_text = "n/a" if overlap_count < 0 else overlap_count
    print(f"{collection:<20}{str(total_text):>14}{str(overlap_text):>14}")

## 4. Extract clipped pixels from Planetary Computer

Search the same Planetary Computer collections over the selected analysis bbox and load COG pixels onto the shared `GEOBOX` at 10 m in the dynamically selected local UTM zone. Every source is reprojected and resampled to the identical grid.

In [ ]:
import planetary_computer as pc
import pystac_client
from odc.stac import load as odc_load

CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)


def search(collection, bbox, datetime=None, query=None):
    items = list(CATALOG.search(
        collections=[collection], bbox=bbox, datetime=datetime, query=query,
    ).items())
    print(f"  {collection:<20} {len(items):>3} item(s)")
    return items


def load(items, bands, resampling="bilinear"):
    """Load items onto the shared GEOBOX without dask."""
    return odc_load(
        items,
        bands=bands,
        geobox=GEOBOX,
        resampling=resampling,
        chunks=None,
    )


print(f"Searching Planetary Computer over the {BUFFER_KM:g} km analysis clip ...")
s2_items = search("sentinel-2-l2a", BBOX_WGS, SEASON, {"eo:cloud_cover": {"lt": 35}})
dem_items = search("cop-dem-glo-30", BBOX_WGS)
s1_items = search("sentinel-1-rtc", BBOX_WGS, SEASON)
wc_items = search("esa-worldcover", BBOX_WGS)

In [ ]:
def _med(ds, dim="time"):
    return ds.median(dim=dim, skipna=True) if dim in ds.dims else ds

# --- Sentinel-2 L2A: surface reflectance (scaled 0-1) -> indices ---
s2 = load(s2_items, ["B02", "B03", "B04", "B08", "B11", "B12"])
s2 = _med(s2) / 10000.0
B02, B03, B04 = s2["B02"], s2["B03"], s2["B04"]
B08, B11, B12 = s2["B08"], s2["B11"], s2["B12"]

# --- Copernicus DEM GLO-30: elevation (m) ---
dem  = load(dem_items, ["data"], resampling="bilinear")
elev = _med(dem)["data"].astype("float32")

# --- Sentinel-1 RTC: linear power -> dB ---
s1 = load(s1_items, ["vv", "vh"])
s1 = _med(s1)
vv_db = 10.0 * np.log10(s1["vv"].where(s1["vv"] > 0))
vh_db = 10.0 * np.log10(s1["vh"].where(s1["vh"] > 0))

# --- ESA WorldCover: categorical land-use (nearest, latest epoch) ---
wc = load(wc_items, ["map"], resampling="nearest")
wc_class = (wc["map"].isel(time=-1) if "time" in wc.dims else wc["map"]).astype("int16")

print("Loaded clipped rasters onto the shared grid:")
for name, da in [("S2 B08", B08), ("DEM", elev), ("S1 vv_db", vv_db), ("WorldCover", wc_class)]:
    print(f"  {name:<12} shape={tuple(da.shape)}  valid={int(np.isfinite(np.asarray(da, dtype='float64')).sum())}")

## 5. RF-1 factor metrics — indices + landform

Per the RF-1 workflow (spec §6) compute the spectral indices (NDVI, NDWI, BSI,
organic-soils ratio), terrain (slope, valley-bottom-flatness proxy) from the
Copernicus DEM, and radar wetness (S1 VV/VH dB). All arrays share the 10 m grid.

In [ ]:
def arr(da):
    return np.asarray(da, dtype="float32")

eps = 1e-6
b2, b3, b4 = arr(B02), arr(B03), arr(B04)
b8, b11, b12 = arr(B08), arr(B11), arr(B12)

# --- spectral indices ---
ndvi = (b8 - b4) / (b8 + b4 + eps)                                  # vegetation
ndwi = (b3 - b8) / (b3 + b8 + eps)                                  # McFeeters water
bsi  = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)    # bare soil
organic_idx = b11 / (b12 + eps)                                     # SWIR1/SWIR2 organic proxy

# --- terrain from DEM (10 m grid) ---
z = arr(elev)
gy, gx = np.gradient(z, RES_M, RES_M)               # dz/dy, dz/dx in m/m
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))

# Valley-bottom-flatness proxy (MrVBF-style): flat + low-lying = high.
def _pct_rank(a):
    f = a[np.isfinite(a)]
    if f.size == 0:
        return np.zeros_like(a)
    order = np.argsort(f)
    ranks = np.empty_like(order, dtype="float32")
    ranks[order] = np.linspace(0.0, 1.0, f.size, dtype="float32")
    out = np.full(a.shape, np.nan, dtype="float32")
    out[np.isfinite(a)] = ranks
    return out

flatness  = np.clip(1.0 - (slope_deg / 12.0), 0.0, 1.0)   # 1 at 0deg, 0 at >=12deg
lowness   = 1.0 - _pct_rank(z)                            # 1 at valley floor
vbf_proxy = np.clip(flatness * lowness, 0.0, 1.0)

# --- radar wetness ---
vv = arr(vv_db)
vh = arr(vh_db)

wcc = np.asarray(wc_class, dtype="int16")

print("Metric ranges (finite):")
for nm, a in [("ndvi", ndvi), ("ndwi", ndwi), ("bsi", bsi), ("organic_idx", organic_idx),
              ("slope_deg", slope_deg), ("vbf_proxy", vbf_proxy), ("vv_db", vv)]:
    f = a[np.isfinite(a)]
    if f.size:
        print(f"  {nm:<12} min={f.min():7.2f}  med={np.median(f):7.2f}  max={f.max():7.2f}")

## 5b. Source ground truth and identity (BC SIFT + surficial geology)

The satellite/DEM/radar metrics above are *indirect* proxies for soft ground. The
bronze **BC Soil Information Finder Tool** polygons (`bronze_bc_soil_survey_polygons`)
are surveyed ground truth: their drainage class / parent material / texture map
directly onto soft-soil behaviour. Each polygon is reprojected to the UTM analysis CRS,
scored from those attributes (`organic`/`very poorly drained`/`gley` -> high,
`fluvial`/`lacustrine`/`clay` -> moderate), and **rasterised onto the 10 m grid** so it
can be blended into Susceptibility.

This step also burns **feature identity** onto the grid. Alongside the soft-soil score,
each pixel records which soil polygon and which surficial-geology polygon it falls in
(`soil_poly_id`, `geology_poly_id`; `0` means no mapped polygon). Without this the
downstream layers can say *how soft* a pixel is but never *which mapped unit* it sits
on - which is exactly what a screening report and a data agent need to cite.

Degrades gracefully to an all-zero signal and empty identity if a bronze table is absent.

In [ ]:
import json
from rasterio.features import rasterize
from rasterio.warp import transform_geom

# BC SIFT stores CODED attributes, not free text: DRAIN_1=W, MDEP_1=COLL, TEXTURE_1=SL.
# A keyword scan for words like "organic" or "poorly drained" matches none of them and
# silently returns the default for every polygon, which turns the survey signal into a
# flat constant. These tables decode the codes that actually appear in the data.
DRAIN_SCORE = {
    "VP": 1.00, "P": 0.90, "I": 0.60, "MW": 0.35, "W": 0.20, "R": 0.10, "VR": 0.05,
}
MDEP_SCORE = {
    "ORGA": 1.00, "ORG": 1.00, "LACU": 0.75, "FLLA": 0.72, "FLUV": 0.70,
    "MARI": 0.70, "GLMA": 0.70, "EOLI": 0.40, "COLL": 0.30, "MORA": 0.30,
    "BEDR": 0.05, "ROCK": 0.05,
}
TEXTURE_SCORE = {
    "C": 0.70, "SIC": 0.70, "SICL": 0.65, "CL": 0.60, "SC": 0.55, "SCL": 0.45,
    "SIL": 0.50, "SI": 0.50, "L": 0.40, "SL": 0.30, "LS": 0.25, "S": 0.20,
}
DRAIN_LABELS = {
    "VP": "very poorly drained", "P": "poorly drained", "I": "imperfectly drained",
    "MW": "moderately well drained", "W": "well drained", "R": "rapidly drained",
    "VR": "very rapidly drained",
}
MDEP_LABELS = {
    "ORGA": "organic", "ORG": "organic", "LACU": "lacustrine", "FLUV": "fluvial",
    "FLLA": "fluvio-lacustrine", "MARI": "marine", "GLMA": "glaciomarine",
    "EOLI": "eolian", "COLL": "colluvial", "MORA": "morainal", "BEDR": "bedrock",
    "ROCK": "bedrock",
}

# Free-text fallback for layers that are not SIFT (surficial geology, etc.).
SOFT_KEYS = {
    "very poorly": 1.0, "organic": 1.0, "peat": 1.0, "muck": 1.0, "bog": 1.0,
    "gleysol": 0.9, "gley": 0.9, "poorly drained": 0.9, "poorly": 0.85,
    "fluvial": 0.7, "alluv": 0.7, "lacustrine": 0.7, "marine": 0.7, "fen": 0.9,
    "imperfectly": 0.6, "clay": 0.6, "silt": 0.5,
}


def _sift_code(properties, key):
    value = properties.get(key)
    if value in (None, "", "-", "None"):
        return None
    return str(value).strip().upper()


def _component_score(properties, index):
    """Strongest soft-ground signal among drainage, parent material, and texture."""
    candidates = [
        DRAIN_SCORE.get(_sift_code(properties, "DRAIN_" + str(index))),
        MDEP_SCORE.get(_sift_code(properties, "MDEP_" + str(index))),
        TEXTURE_SCORE.get(_sift_code(properties, "TEXTURE_" + str(index))),
    ]
    present = [score for score in candidates if score is not None]
    return max(present) if present else None


def soft_soil_score_from_properties(properties):
    """Component-percent-weighted soft-ground score in [0,1].

    A SIFT polygon carries up to three soil components with a PERCENT_n share each, so
    the polygon score is the area-weighted mean of its components rather than a single
    lookup. Falls back to a keyword scan for non-SIFT layers.
    """
    total = 0.0
    weight = 0.0
    for index in (1, 2, 3):
        score = _component_score(properties, index)
        if score is None:
            continue
        percent = properties.get("PERCENT_" + str(index))
        try:
            share = float(percent) if percent not in (None, "", "None") else 0.0
        except (TypeError, ValueError):
            share = 0.0
        if share <= 0:
            share = 1.0
        total += score * share
        weight += share
    if weight > 0:
        return round(total / weight, 4)

    blob = " ".join(str(value) for value in properties.values()).lower()
    best = 0.0
    for keyword, value in SOFT_KEYS.items():
        if keyword in blob:
            best = max(best, value)
    return best if best > 0 else 0.4


def _soft_score(props_json):
    """Soft-ground score for one bronze feature, from its raw properties JSON."""
    try:
        properties = json.loads(props_json) if props_json else {}
    except Exception:
        return 0.4
    if not isinstance(properties, dict):
        return 0.4
    return soft_soil_score_from_properties(properties)


def load_vector_layer(table_name):
    """Bronze GeoJSON -> [(feature_id, geometry_in_utm, properties_json)]."""
    features = []
    try:
        rows = (spark.read.format("delta").load(f"{BRONZE_ABFSS}/{table_name}")
                .select("feature_id", "geometry_json", "properties_json").collect())
    except Exception as error:
        print(f"  (skip {table_name}: {str(error)[:100]})")
        return features
    for row in rows:
        try:
            geometry = json.loads(row["geometry_json"]) if row["geometry_json"] else None
            if not geometry:
                continue
            features.append((row["feature_id"],
                             transform_geom("EPSG:4326", UTM_CRS, geometry),
                             row["properties_json"]))
        except Exception:
            continue
    print(f"  {table_name}: {len(rows)} bronze rows -> {len(features)} usable geometries")
    return features


aff = GEOBOX.affine
lookup_rows = []


def burn_identity(features, layer_name):
    """Burn a 1-based polygon index onto the grid; 0 means 'no mapped polygon here'.

    Overlapping polygons resolve last-one-wins, which is rasterio's documented
    behaviour and acceptable for a screening-grade demo.
    """
    if not features:
        return np.zeros((H, W), dtype="int32")
    identity = rasterize(
        [(geometry, index + 1) for index, (_, geometry, _) in enumerate(features)],
        out_shape=(H, W), transform=aff, fill=0, all_touched=True, dtype="int32",
    )
    for index, (feature_id, _, properties_json) in enumerate(features):
        lookup_rows.append({
            "run_id": RUN_ID,
            "layer": layer_name,
            "poly_id": index + 1,
            "feature_key": f"{layer_name}:{feature_id}",
            "feature_id": feature_id,
            "soft_soil_score": float(_soft_score(properties_json)),
        })
    return identity


# --- BC Soil Information Finder Tool (SIFT) polygons: signal + identity ---
soil_features = load_vector_layer("bronze_bc_soil_survey_polygons")
if soil_features:
    soil_soft = rasterize(
        [(geometry, _soft_score(properties)) for _, geometry, properties in soil_features],
        out_shape=(H, W), transform=aff, fill=0.0, all_touched=True, dtype="float32",
    )
    soil_mapped = rasterize(
        [(geometry, 1) for _, geometry, _ in soil_features],
        out_shape=(H, W), transform=aff, fill=0, all_touched=True, dtype="uint8",
    ).astype("float32")
else:
    print("  no soil polygons over the AOI clip -> soil signal is all-zero")
    soil_soft = np.zeros((H, W), dtype="float32")
    soil_mapped = np.zeros((H, W), dtype="float32")
soil_poly_id = burn_identity(soil_features, "soil_survey")

# --- BC quaternary / surficial geology: identity only (context for the report) ---
geology_features = load_vector_layer("bronze_bc_quaternary_geology")
geology_poly_id = burn_identity(geology_features, "surficial_geology")

print(f"Soil-survey ground truth burned onto grid: "
      f"mapped={int(soil_mapped.sum()):,} px, soft>0={int((soil_soft > 0).sum()):,} px")
print(f"Source identity burned onto grid: "
      f"soil={int((soil_poly_id > 0).sum()):,} px, geology={int((geology_poly_id > 0).sum()):,} px")

## 6. Susceptibility (S) and Consequence (C) ratings

- **S (1–5)** — soft-soil susceptibility blended from the **BC soil-survey ground
  truth** (largest weight), the valley-bottom-flatness proxy, radar/optical wetness,
  and bare/organic signals (spec §7). WorldCover wetland classes (90/95) with S1
  VV < −15 dB, and any pixel the survey maps as organic / very-poorly-drained, are
  forced to the top of the scale.
- **C (1–5)** — consequence/exposure proxy from WorldCover land-use (built-up →
  high, water → low), nudged up on steep slopes where movement matters (spec §8).


In [ ]:
def nz(a):
    return np.nan_to_num(a, nan=0.0)

def clip01(a):
    return np.clip(a, 0.0, 1.0)

# --- soft-soil probability P (0..1): availability-weighted blend ---
sig_vbf     = clip01(vbf_proxy)                       # valley-bottom flatness
sig_soil    = clip01(soil_soft)                       # BC soil-survey ground truth
is_wetland  = np.isin(wcc, [90, 95]).astype("float32")
is_water    = (wcc == 80).astype("float32")
wet_radar   = clip01((-(vv) - 12.0) / 8.0)            # VV<-12 -> wet, VV<-20 -> 1
wet_ndwi    = clip01(ndwi * 2.0)
bare        = clip01((bsi + 0.10) / 0.40) * (ndvi < 0.30)
organic     = clip01((organic_idx - 1.0) / 0.5)

# BC SIFT survey is direct evidence, so it carries the single largest weight.
p_soft = clip01(
    0.25 * nz(sig_vbf) +
    0.20 * nz(sig_soil) +
    0.15 * (0.5 * is_wetland + 0.5 * is_water) +
    0.13 * nz(wet_radar) +
    0.12 * nz(wet_ndwi) +
    0.08 * nz(bare) +
    0.07 * nz(organic)
)
# where the survey maps organic / very-poorly-drained ground, lift P to the top
p_soft = np.where(soil_soft >= 0.9, np.maximum(p_soft, 0.85), p_soft)

# --- S rating 1..5 ---
s_rating = np.digitize(p_soft, [0.20, 0.40, 0.60, 0.80]).astype("int16") + 1
s_rating = np.where((is_wetland == 1) & (vv < -15.0), 5, s_rating)
s_rating = np.where((is_wetland == 1) & (s_rating < 4), 4, s_rating)
s_rating = np.where(is_water == 1, 5, s_rating)
s_rating = np.where(soil_soft >= 0.9, np.maximum(s_rating, 4), s_rating)  # surveyed soft soil
s_rating = np.clip(s_rating, 1, 5).astype("int16")

# --- C rating 1..5 from WorldCover exposure, modulated by slope ---
exposure = {10: 2, 20: 3, 30: 3, 40: 4, 50: 5, 60: 2, 70: 1, 80: 1, 90: 2, 95: 2, 100: 1}
c_base = np.vectorize(lambda v: exposure.get(int(v), 2))(wcc).astype("int16")
c_rating = np.where(slope_deg > 15.0, c_base + 1, c_base)
c_rating = np.clip(c_rating, 1, 5).astype("int16")

print("S rating distribution:", {int(k): int(v) for k, v in zip(*np.unique(s_rating, return_counts=True))})
print("C rating distribution:", {int(k): int(v) for k, v in zip(*np.unique(c_rating, return_counts=True))})


## 7. Write the silver table

Flatten the clipped grid to one row per 10 m pixel (with geographic coordinates,
all factor metrics and the S/C ratings) and persist as the Delta table
`silver_rf1_soil_susceptibility` in the silver lakehouse.

In [ ]:
import pandas as pd

# pixel-centre coordinates from the geobox affine
aff = GEOBOX.affine
rows, cols = np.mgrid[0:H, 0:W]
utm_x = aff.c + (cols + 0.5) * aff.a + (rows + 0.5) * aff.b
utm_y = aff.f + (cols + 0.5) * aff.d + (rows + 0.5) * aff.e
lon_g, lat_g = _to_wgs.transform(utm_x, utm_y)

now_utc = datetime.now(timezone.utc).isoformat()

pdf = pd.DataFrame({
    "row":             rows.ravel().astype("int32"),
    "col":             cols.ravel().astype("int32"),
    "utm_x":           utm_x.ravel().astype("float64"),
    "utm_y":           utm_y.ravel().astype("float64"),
    "lon":             lon_g.ravel().astype("float64"),
    "lat":             lat_g.ravel().astype("float64"),
    "elevation_m":     z.ravel().astype("float32"),
    "slope_deg":       slope_deg.ravel().astype("float32"),
    "vbf_proxy":       vbf_proxy.ravel().astype("float32"),
    "ndvi":            ndvi.ravel().astype("float32"),
    "ndwi":            ndwi.ravel().astype("float32"),
    "bsi":             bsi.ravel().astype("float32"),
    "organic_idx":     organic_idx.ravel().astype("float32"),
    "vv_db":           vv.ravel().astype("float32"),
    "vh_db":           vh.ravel().astype("float32"),
    "worldcover_class": wcc.ravel().astype("int16"),
    "soil_soft":       soil_soft.ravel().astype("float32"),
    "soil_mapped":     soil_mapped.ravel().astype("int16"),
    "soil_poly_id":    soil_poly_id.ravel().astype("int32"),
    "geology_poly_id": geology_poly_id.ravel().astype("int32"),
    "p_soft":          p_soft.ravel().astype("float32"),
    "s_rating":        s_rating.ravel().astype("int16"),
    "c_rating":        c_rating.ravel().astype("int16"),
})

# keep pixels with valid terrain (inside the DEM footprint)
pdf = pdf[np.isfinite(pdf["elevation_m"])].reset_index(drop=True)
pdf["aoi_name"]       = AOI_NAME
pdf["aoi_lat"]        = LAT
pdf["aoi_lon"]        = LON
pdf["buffer_km"]      = BUFFER_KM
pdf["resolution_m"]   = RES_M
pdf["risk_factor"]    = "RF-1"
pdf["ingested_at_utc"] = now_utc
pdf["run_id"]          = RUN_ID

# --- make the frame Spark/Delta-safe (fixes TASK_WRITE_FAILED on saveAsTable) ---
# The OneLake write task fails on: numpy int16/float32 columns, NaN/Inf in float
# metrics, and the Arrow columnar write path. Normalise every column to a
# Spark-friendly dtype and disable the Arrow fast-path so the write is deterministic.
for c in pdf.select_dtypes(include=["float16", "float32", "float64"]).columns:
    pdf[c] = np.nan_to_num(pdf[c].astype("float64"), nan=0.0, posinf=0.0, neginf=0.0)
for c in pdf.select_dtypes(include=["int8", "int16", "int64",
                                    "uint8", "uint16", "uint32", "uint64"]).columns:
    pdf[c] = pdf[c].astype("int32")

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

print(f"Silver pixels: {len(pdf):,}  ({H}x{W} grid, {RES_M} m)")
print(f"  soil-mapped pixels kept: {int(pdf['soil_mapped'].sum()):,}")

sdf = spark.createDataFrame(pdf)


def write_run_scoped(dataframe, table_name):
    """Dynamic partition overwrite: replace only this run, keep earlier runs."""
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
            .saveAsTable(table_name))
    except Exception as error:
        # A schema change cannot be applied in dynamic partition overwrite mode:
        # Delta rejects overwriteSchema with DELTA_OVERWRITE_SCHEMA_WITH_DYNAMIC_
        # PARTITION_OVERWRITE. Drop back to a static full overwrite, which replaces
        # every run, then restore dynamic mode for subsequent writes.
        print(f"  WARNING: {table_name} schema changed - replacing ALL runs. "
              f"({str(error).splitlines()[0][:120]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


write_run_scoped(sdf, "silver_rf1_soil_susceptibility")

# Polygon lookup: maps the burned soil_poly_id / geology_poly_id back to the bronze
# feature_key, which joins to silver_source_features for names and attributes.
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

LOOKUP_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("poly_id", IntegerType(), True),
    StructField("feature_key", StringType(), True),
    StructField("feature_id", StringType(), True),
    StructField("soft_soil_score", DoubleType(), True),
])
_lookup_columns = [field.name for field in LOOKUP_SCHEMA.fields]
_lookup_records = [tuple(row.get(column) for column in _lookup_columns) for row in lookup_rows]
lookup_sdf = spark.createDataFrame(_lookup_records, schema=LOOKUP_SCHEMA)
write_run_scoped(lookup_sdf, "silver_rf1_poly_lookup")
print(f"Wrote silver_rf1_poly_lookup -> {lookup_sdf.count():,} polygons")

cnt = spark.read.table("silver_rf1_soil_susceptibility").count()
print(f"Wrote silver_rf1_soil_susceptibility -> {cnt:,} rows")
